In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
import statsmodels.api as sm

In [ ]:
data = pd.read_csv("final_dataset.csv")

In [ ]:
df = pd.DataFrame(data)
df.head()

,heart,lungs,liver,kidneys,stomach,spine,diabetes,hypertension,joints,ENT_organs,...,invalid,mar_st,visit_doctor,work,alcohol,smoking,phys_active,is_health_good,is_health_very_good,diploma
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,...,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0


In [ ]:
params = [
    "age",
    "income",
    "n_child",
    "sex",
    "type_area",
    "invalid",
    "mar_st",
    "visit_doctor",
    "work",
    "alcohol",
    "smoking",
    "phys_active",
    "is_health_good",
    "is_health_very_good",
    "diploma",
]
targets = df.keys().drop(params).to_list()  # проведем эту операцию еще раз

In [ ]:
df[params]

,age,income,n_child,sex,type_area,invalid,mar_st,visit_doctor,work,alcohol,smoking,phys_active,is_health_good,is_health_very_good,diploma
0,44.0,43000.0,2.0,2.0,0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0
1,61.0,20000.0,3.0,2.0,0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
2,55.5,45000.0,3.0,1.0,0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
3,40.5,50000.0,2.0,2.0,0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0
4,55.0,55000.0,1.0,1.0,0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4593,50.5,90000.0,4.0,2.0,0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0
4594,37.5,60000.0,2.0,2.0,0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
4595,48.0,20000.0,2.0,1.0,0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4596,37.5,20000.0,2.0,2.0,0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
df_men = df[df["sex"] == 1]
df_women = df[df["sex"] == 2]

In [ ]:
def perform_matching(data, treatment_col, outcomes, covars, method="psm"):
    data_clean = data.dropna(subset=covars + [treatment_col])
    X = data_clean[covars]
    T = data_clean[treatment_col]
    results = {}

    if method == "psm":
        lr = LogisticRegression()
        lr.fit(X, T)
        data_clean["ps"] = lr.predict_proba(X)[:, 1]
        treated = data_clean[data_clean[treatment_col] == 1]
        untreated = data_clean[data_clean[treatment_col] == 0]
        nn = NearestNeighbors(n_neighbors=1).fit(untreated[["ps"]])
        _, indices = nn.kneighbors(treated[["ps"]])
        matched_data = pd.concat([treated, untreated.iloc[indices.flatten()]])
    else:
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        X_scaled_df = pd.DataFrame(X_scaled, index=data_clean.index, columns=covars)
        treated_scaled = X_scaled_df[data_clean[treatment_col] == 1]
        untreated_scaled = X_scaled_df[data_clean[treatment_col] == 0]
        vi = np.linalg.pinv(np.cov(X_scaled.T))
        nn = NearestNeighbors(
            n_neighbors=1, metric="mahalanobis", metric_params={"VI": vi}
        ).fit(untreated_scaled)
        _, indices = nn.kneighbors(treated_scaled)
        matched_data = pd.concat([
            data_clean[data_clean[treatment_col] == 1],
            data_clean.loc[untreated_scaled.index[indices.flatten()]],
        ])

    for outcome in outcomes:
        if matched_data[outcome].nunique() > 1:
            try:
                model = sm.Logit(
                    matched_data[outcome], sm.add_constant(matched_data[treatment_col])
                ).fit(disp=0)
                results[outcome] = {
                    "pval": model.pvalues[treatment_col],
                    "effect": model.params[treatment_col],
                }
            except:
                results[outcome] = {"pval": np.nan, "effect": np.nan}
        else:
            results[outcome] = {"pval": np.nan, "effect": np.nan}
    return results


treatment = "diploma"
covariates = [
    "age",
    "income",
    "n_child",
    "type_area",
    "smoking",
    "alcohol",
    "phys_active",
]

final_tables = {}
for g_name, g_df in [("Men", df_men), ("Women", df_women)]:
    psm_res = perform_matching(g_df, treatment, targets, covariates, method="psm")
    mah_res = perform_matching(
        g_df, treatment, targets, covariates, method="mahalanobis"
    )

    combined = pd.DataFrame({
        "Disease": targets,
        "PSM Effect": [psm_res.get(t)["effect"] for t in targets],
        "PSM P-Value": [psm_res.get(t)["pval"] for t in targets],
        "Mahalanobis Effect": [mah_res.get(t)["effect"] for t in targets],
        "Mahalanobis P-Value": [mah_res.get(t)["pval"] for t in targets],
    })
    combined["PSM Sig"] = combined["PSM P-Value"] < 0.05
    combined["Mah Sig"] = combined["Mahalanobis P-Value"] < 0.05
    final_tables[g_name] = combined

for title, table in final_tables.items():
    print(f"\n--- {title} ---")
    display(table)


--- Men ---


,Disease,PSM Effect,PSM P-Value,Mahalanobis Effect,Mahalanobis P-Value,PSM Sig,Mah Sig
0,heart,0.416155,0.104184,1.024812,0.000895,False,True
1,lungs,-0.222308,0.457612,-0.440529,0.124825,False,False
2,liver,0.760887,0.050269,0.663869,0.078091,False,False
3,kidneys,0.294555,0.445824,-0.062350,0.859892,False,False
4,stomach,0.159884,0.349047,0.289504,0.098805,False,False
5,spine,-0.399587,0.021315,-0.034079,0.853549,True,False
6,diabetes,1.168772,0.004404,0.756481,0.033634,True,True
7,hypertension,0.581554,0.000238,0.467174,0.002483,True,True
8,joints,0.308402,0.097572,0.270199,0.143200,False,False
9,ENT_organs,0.187607,0.454236,1.088006,0.000903,False,True



--- Women ---


,Disease,PSM Effect,PSM P-Value,Mahalanobis Effect,Mahalanobis P-Value,PSM Sig,Mah Sig
0,heart,0.155513,0.495367,0.127975,0.572118,False,False
1,lungs,-0.826720,0.000010,-0.732403,0.000110,True,True
2,liver,0.383314,0.171084,0.212821,0.425889,False,False
3,kidneys,-0.280331,0.149352,-0.462925,0.013804,False,True
4,stomach,-0.038088,0.735358,-0.117395,0.291270,False,False
5,spine,-0.352449,0.002633,-0.399808,0.000593,True,True
6,diabetes,-0.235283,0.136334,-0.141896,0.377654,False,False
7,hypertension,-0.075518,0.467299,-0.152802,0.137016,False,False
8,joints,-0.234105,0.046680,-0.155854,0.190993,True,False
9,ENT_organs,-0.232647,0.110880,-0.132390,0.373328,False,False


### Анализ результатов

Выше представлена часть из 60 моделей (15 заболеваний × 2 пола × 2 метода).

1. **Propensity Score Matching (PSM)**: Позволяет сбалансировать группы образованных и необразованных по ковариатам (возраст, доход и т.д.) на основе вероятности получения воздействия.
2. **Mahalanobis Distance**: Проводит поиск «близнецов» на основе многомерного расстояния.

Если `P-Value < 0.05`, это означает, что после выравнивания групп выбранный фактор сохраняет статистически значимую связь с конкретным заболеванием у данного пола.